In [2]:
! pip install transformers torch accelerate


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_PATH = r"C:\qwen1.5b"
SYSTEM_PROMPT = "You are a helpful technical assistant. Answer clearly and accurately."
MAX_TOKENS = 512

print("Loading model... (1-2 minutes)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,   # CPU
    device_map="cpu",
)
model.eval()

print("=" * 40)
print("   Qwen2.5-1.5B  |  Offline  |  CPU")
print("   'clear' = new chat  |  'quit' = exit")
print("=" * 40 + "\n")

history = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(user_input):
    history.append({"role": "user", "content": user_input})

    prompt = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": response})
    return response


while True:
    try:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "bye"):
            print("Bye!")
            break
        if user_input.lower() == "clear":
            history.clear()
            history.append({"role": "system", "content": SYSTEM_PROMPT})
            print("--- Chat cleared ---\n")
            continue
        print("Thinking...\n")
        reply = chat(user_input)
        print(f"Bot: {reply}\n")
        print("-" * 40)
    except KeyboardInterrupt:
        print("\nBye!")
        break